# ‖Δ‖, C0 probability, confidence and C2 scores

* **dist** = ‖Δ‖₂ between x_i and its nearest CF, fold-standardised attribute space (cached by `cf_distance.ipynb`).
* **prob** = raw C0 probability; **conf** = |p − t|, distance from C0's Youden threshold.
* **attr** = full attribute model (MLP I+C+D, K=10); **cnn** = full CNN (dual_sal, K=10). Out-of-fold scores.

‖Δ‖ has extreme outliers (effusion max ≈ 7e6 vs median 14), so Pearson uses log‖Δ‖; Spearman is rank-based.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

RES = '../results'
R = '/work3/s251710/thesis_results'
DISEASES = ['effusion', 'cardiomegaly', 'consolidation', 'edema', 'atelectasis']

def oof(d, sub, col):
    return pd.concat([pd.read_csv(f'{R}/C2_final_results/{d}/knn_correct_cf/cf_10/{sub}/fold_{i}_predictions.csv',
                                  usecols=['path', col]) for i in range(5)])

dfs = []
for d in DISEASES:
    df = pd.read_csv(f'{RES}/cf_distance_l2_{d}.csv').rename(columns={f'{d}_prob': 'prob', 'cf_dist': 'dist'})
    t = float(open(f'{R}/C0_final/multilabel_ignore/{d}/threshold.txt').read())
    df['conf'] = (df.prob - t).abs()
    df = (df.merge(oof(d, 'attributes', 'MLP_I+C+D_prob').rename(columns={'MLP_I+C+D_prob': 'attr'}), on='path')
            .merge(oof(d, 'images/dual_sal', 'dual_sal_prob').rename(columns={'dual_sal_prob': 'cnn'}), on='path'))
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
df['log_dist'] = np.log(df.dist)
df.groupby('disease').size()

disease
atelectasis      80938
cardiomegaly     92507
consolidation    83688
edema            89852
effusion         90934
dtype: int64

In [2]:
PAIRS = [('dist', 'prob'), ('dist', 'conf'), ('attr', 'prob'), ('cnn', 'prob'), ('attr', 'cnn')]

rows = []
for d, g in df.groupby('disease', sort=False):
    for a, b in PAIRS:
        rows.append({'disease': d, 'pair': f'{a}~{b}',
                     'spearman': spearmanr(g[a], g[b])[0],
                     'pearson': pearsonr(g['log_dist' if a == 'dist' else a], g[b])[0]})
pd.DataFrame(rows).pivot(index='disease', columns='pair', values=['spearman', 'pearson']).loc[DISEASES].round(3)

spearman                                       pearson           \
pair          attr~cnn attr~msp cnn~msp dist~conf dist~prob attr~cnn attr~msp   
disease                                                                         
effusion         0.882    0.941   0.896    -0.004     0.197    0.882    0.894   
cardiomegaly     0.865    0.922   0.861    -0.035    -0.055    0.958    0.920   
consolidation    0.892    0.923   0.886     0.014     0.057    0.985    0.992   
edema            0.946    0.951   0.941     0.080     0.112    0.945    0.911   
atelectasis      0.907    0.927   0.922    -0.057    -0.068    0.970    0.972   

                                           
pair          cnn~msp dist~conf dist~prob  
disease                                    
effusion        0.864    -0.005     0.176  
cardiomegaly    0.910    -0.051    -0.066  
consolidation   0.982     0.063     0.083  
edema           0.915     0.075     0.112  
atelectasis     0.967    -0.068    -0.080